# 5ifthNN — One-Click Google Colab L4 Training & ONNX Export

**How to use (Just click `Runtime → Run All`):**
1. Upload your `5ifthNN_dataset_*.zip` (or `.5nn.gz` files) into Google Drive: `MyDrive/5ifthNN/recordings/`.
2. Click **`Runtime → Run All`**.
3. Download `fifthnn_v1.onnx` (and `fifthnn_v1_int8.onnx`) from `MyDrive/5ifthNN/models/` and put it in `.minecraft/config/5ifthnn/models/`.

*(Optional Phase 2B PPO: Once you have AI self-play/bot-play `.5nn.gz` files, drop them into `MyDrive/5ifthNN/selfplay_rollouts/` and click **Run All** again — PPO runs automatically!)*

In [ ]:
# 1. Mount Google Drive, sync 5ifthNN repo & install dependencies
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!git clone https://github.com/5ivesaw/5ifthNN.git /content/5ifthNN || (cd /content/5ifthNN && git pull)
%cd /content/5ifthNN/python
!pip -q install -r requirements.txt

In [ ]:
# 2. Auto-unpack any uploaded .zip bundles (combinable from multiple players!) & Audit
import glob, os, zipfile
from pathlib import Path

DATA_DIR = '/content/drive/MyDrive/5ifthNN/recordings'
ROLLOUT_DIR = '/content/drive/MyDrive/5ifthNN/selfplay_rollouts'
CKPT_DIR = '/content/drive/MyDrive/5ifthNN/checkpoints'
MODEL_DIR = '/content/drive/MyDrive/5ifthNN/models'
UNPACK_DIR = '/content/fifthnn_unpacked_recordings'

for d in (DATA_DIR, ROLLOUT_DIR, CKPT_DIR, MODEL_DIR, UNPACK_DIR):
    os.makedirs(d, exist_ok=True)

# Unpack any .zip archives from DATA_DIR into UNPACK_DIR (supports combining multiple zips!)
zip_files = sorted(glob.glob(f'{DATA_DIR}/**/*.zip', recursive=True))
for zf in zip_files:
    print(f'Unpacking {Path(zf).name}...')
    with zipfile.ZipFile(zf, 'r') as z:
        for member in z.namelist():
            if member.endswith(('.5nn.gz', '.5nn', '.meta.json', '.tags.jsonl')):
                out = Path(UNPACK_DIR) / Path(member).name
                if not out.exists():
                    out.write_bytes(z.read(member))

# Also symlink any loose .5nn.gz files directly in DATA_DIR
for f in glob.glob(f'{DATA_DIR}/**/*.5nn*', recursive=True):
    dst = Path(UNPACK_DIR) / Path(f).name
    if not dst.exists():
        os.symlink(f, dst)

!python3 -m fifthnn.audit "$UNPACK_DIR"

In [ ]:
# 3. Phase 2A: Behavior Cloning (BC) Bootstrap -> starter brain
!python3 -m fifthnn.train_bc \
  --data_dir "$UNPACK_DIR" \
  --out "$CKPT_DIR/fifthnn_bc_v1.pt" \
  --epochs 25 \
  --batch_size 32 \
  --seq_len 128 \
  --lr 3e-4

In [ ]:
# 4. Phase 2B (Auto-runs if selfplay_rollouts has .5nn.gz files): KL-Anchored PPO
import glob, subprocess

rollouts = [p for p in glob.glob(f'{ROLLOUT_DIR}/**/*.5nn*', recursive=True) if p.endswith(('.5nn', '.5nn.gz'))]
BC_CKPT = f'{CKPT_DIR}/fifthnn_bc_v1.pt'
PPO_CKPT = f'{CKPT_DIR}/fifthnn_ppo_v1.pt'

if rollouts:
    print(f'Found {len(rollouts)} AI rollout recording(s) -> Running KL-Anchored PPO!')
    subprocess.run([
        'python3', '-m', 'fifthnn.train_ppo',
        '--rollout_dir', ROLLOUT_DIR,
        '--bc_anchor', BC_CKPT,
        '--policy', PPO_CKPT if os.path.exists(PPO_CKPT) else BC_CKPT,
        '--out', PPO_CKPT,
        '--kl_coef', '0.08',
        '--epochs', '4',
    ], check=True)
    EXPORT_CKPT = PPO_CKPT
else:
    print('No AI rollouts in selfplay_rollouts/ yet -> Skipping PPO and exporting BC starter brain.')
    EXPORT_CKPT = BC_CKPT

In [ ]:
# 5. Export to ONNX (FP32 + INT8) & Verify Parity + CPU Latency
!python3 -m fifthnn.export_onnx \
  --ckpt "$EXPORT_CKPT" \
  --out "$MODEL_DIR/fifthnn_v1.onnx" \
  --quantize_int8

print(f'DONE! Download fifthnn_v1.onnx from Google Drive -> MyDrive/5ifthNN/models/')
print('Put it in .minecraft/config/5ifthnn/models/ alongside 5ifthNN-OnnxRuntime-1.32.0.jar in mods/')